## Step 0: Import the Libraries

We import everything we need in one place, at the top.

In [ ]:
# pandas: to read and work with tables
import pandas as pd

# train_test_split: to split rows into a training part and a test part
from sklearn.model_selection import train_test_split

# LogisticRegression: the model we will train
from sklearn.linear_model import LogisticRegression

# Functions to score the model
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

## Step 1: Load and Look at the Data

Before changing anything, we read the file and look at it.

In [ ]:
# The file is stored on GitHub. pandas can read it straight from the link.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

In [ ]:
df.shape

(7043, 21)

In [ ]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# The 8 input columns we will use
feature_cols = [
    "tenure",           # months the customer has been with the company
    "MonthlyCharges",   # amount billed every month
    "TotalCharges",     # total amount billed so far
    "SeniorCitizen",    # 1 if the customer is a senior citizen, else 0
    "Contract",         # Month-to-month, One year or Two year
    "InternetService",  # DSL, Fiber optic or No
    "PaymentMethod",    # how the customer pays
    "TechSupport",      # Yes, No or No internet service
]

In [ ]:
# The column we want to predict
target_col = "Churn"

# Keep the 8 inputs plus the target
df = df[feature_cols + [target_col]]

df.head()

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.5,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes


### 1.5 How Many Customers Left?

In [ ]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


### 1.6 Does the Contract Type Matter?

In [ ]:
# df.groupby('Contract')['Churn'].value_counts()
df.groupby('Contract')['Churn'].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

**Result.** About 43 percent of month-to-month customers left, against about 3 percent of two-year customers. Contract type will be one of the strongest inputs.

## Step 2: Find Broken or Missing Values

We check two things: blank cells, and columns stored in the wrong type.

In [ ]:
# Code for identifying missing values?
df.isna().sum()

,0
tenure,0
MonthlyCharges,0
TotalCharges,0
SeniorCitizen,0
Contract,0
InternetService,0
PaymentMethod,0
TechSupport,0
Churn,0


**Result.** Every count is zero. It looks clean, but we are not done.

### 2.2 Check the Type of Each Column

In [ ]:
df.info()
# df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tenure           7043 non-null   int64  
 1   MonthlyCharges   7043 non-null   float64
 2   TotalCharges     7043 non-null   object 
 3   SeniorCitizen    7043 non-null   int64  
 4   Contract         7043 non-null   object 
 5   InternetService  7043 non-null   object 
 6   PaymentMethod    7043 non-null   object 
 7   TechSupport      7043 non-null   object 
 8   Churn            7043 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 495.3+ KB


In [ ]:
total_as_number = pd.to_numeric(df['TotalCharges'], errors='coerce') #Suppose column contains "abcd", "NaN"

In [ ]:
# How many values could not be converted?
total_as_number.isna().sum()

np.int64(11)

In [ ]:
df[total_as_number.isna()]

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
488,0,52.55,,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,,0,Two year,DSL,Mailed check,No,No
1082,0,25.75,,0,Two year,No,Mailed check,No internet service,No
1340,0,56.05,,0,Two year,DSL,Credit card (automatic),Yes,No
3331,0,19.85,,0,Two year,No,Mailed check,No internet service,No
3826,0,25.35,,0,Two year,No,Mailed check,No internet service,No
4380,0,20.00,,0,Two year,No,Mailed check,No internet service,No
5218,0,19.70,,0,One year,No,Mailed check,No internet service,No
6670,0,73.35,,0,Two year,DSL,Mailed check,Yes,No


In [ ]:
# We have 11 people where total charge is 0 and tenure is 0
# I am making a small dataset, where I take 7 people from above and 3 more random people with tenure 0
# 7 normal rows from the top, plus 3 of the new customers with tenure 0
small = pd.concat([df.head(7), df[df["tenure"] == 0].head(3)])

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.5,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.5,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.4,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,,0,Two year,DSL,Mailed check,No,No


In [ ]:
# The blank spaces become NaN, so we can see them
small["TotalCharges"] = pd.to_numeric(small["TotalCharges"], errors="coerce")

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.50,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.40,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,NaN,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,NaN,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,NaN,0,Two year,DSL,Mailed check,No,No


In [ ]:
# A new customer has paid nothing yet, so 0 is the correct value
small["TotalCharges"] = small["TotalCharges"].fillna(0)

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.50,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.40,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,0.00,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,0.00,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,0.00,0,Two year,DSL,Mailed check,No,No


In [ ]:
# Same two lines as above, now on the full table
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
df.isna().sum()

,0
tenure,0
MonthlyCharges,0
TotalCharges,0
SeniorCitizen,0
Contract,0
InternetService,0
PaymentMethod,0
TechSupport,0
Churn,0


## Step 4: Prepare the Columns

The model reads only numbers. We turn the target into 1 and 0, and turn the text columns into 0/1 columns.

In [ ]:
# Yes becomes 1, No becomes 0
small["Churn"] = small["Churn"].map({"Yes": 1, "No": 0})

small[["Churn"]]

,Churn
0,0
1,0
2,1
3,0
4,1
5,1
6,0
488,0
753,0
936,0


In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Check: we should see only 0 and 1
df["Churn"].value_counts()

,count
Churn,
0,5174
1,1869


In [ ]:
# Look at 5 rows of the sample before the change
small[feature_cols].head(5)

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No


In [ ]:
# get_dummies makes one 0/1 column for each category of each text column.
# drop_first=True drops one column per text column, because it can be worked out from the others.
# dtype=int shows 0 and 1 instead of False and True.
pd.get_dummies(small[feature_cols].head(5), drop_first=True, dtype=int)

# 3 categories -> Low Medium High
# Low Medium
#  1    0     -> Low
#   0   1     -> Medium
#   0   0     -> High

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract_One year,InternetService_Fiber optic,PaymentMethod_Electronic check,PaymentMethod_Mailed check,TechSupport_Yes
0,1,29.85,29.85,0,0,0,1,0,0
1,34,56.95,1889.50,0,1,0,0,1,0
2,2,53.85,108.15,0,0,0,0,1,0
3,45,42.30,1840.75,0,1,0,0,0,1
4,2,70.70,151.65,0,0,1,1,0,0


**Result.** `Contract` has become `Contract_One year` and `Contract_Two year`. A month-to-month customer has 0 in both. The number columns stay as they are.

### 4.4 Build the Inputs (X) and the Target (y) for All Rows

In [ ]:
# X: the inputs, with text columns turned into 0/1 columns
X = pd.get_dummies(df[feature_cols], drop_first=True, dtype=int)

# y: the answer we want to predict
y = df["Churn"]

X.shape

(7043, 13)

In [ ]:
X.columns

Index(['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
       'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic',
       'InternetService_No', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'TechSupport_No internet service', 'TechSupport_Yes'],
      dtype='object')

## Step 5: Split into Train and Test

The model learns from 75 percent of the rows. We keep 25 percent aside as the final exam.

In [ ]:
# test_size=0.25: one quarter of the rows go to the test set
# random_state=42: everyone in the room gets the same split
# stratify=y: the share of leavers is the same in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 5282
Test rows: 1761


In [ ]:
round(y_train.mean(), 3)

np.float64(0.265)

In [ ]:
round(y_test.mean(), 3)

np.float64(0.265)

In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
# Logistic Regression assigns a weight to all features depending upon their importance for prediction
weights = pd.Series(model.coef_[0], index=X.columns)

weights.sort_values()

,0
Contract_Two year,-1.202669
Contract_One year,-0.744600
InternetService_No,-0.500386
TechSupport_No internet service,-0.500386
TechSupport_Yes,-0.417447
PaymentMethod_Credit card (automatic),-0.133493
tenure,-0.053412
TotalCharges,0.000277
MonthlyCharges,0.001400
PaymentMethod_Mailed check,0.085541


In [ ]:
X_test

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TechSupport_No internet service,TechSupport_Yes
5909,52,80.20,4297.60,0,1,0,1,0,0,0,1,0,0
3670,33,24.25,838.50,0,0,0,0,1,0,0,0,1,0
6220,10,53.70,521.00,0,0,1,0,0,1,0,0,0,1
5905,1,85.00,85.00,0,0,0,1,0,0,1,0,0,0
6435,52,50.20,2554.00,0,0,1,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
476,35,76.05,2747.20,1,0,0,0,0,0,1,0,0,1
1607,13,89.05,1169.35,1,0,0,1,0,0,0,0,0,1
6808,39,100.45,3801.70,0,0,1,1,0,0,1,0,0,0
2962,3,31.00,95.05,0,0,0,0,0,0,0,1,0,1


In [ ]:
scores = model.predict_proba(X_test)[:, 1] # Model throws the probability of both -> Class 0 and Class 1

scores[2]

# Chances of churn

np.float64(0.07729034263898779)

In [ ]:
y_pred = model.predict(X_test)
# Cutoff the model is using currently? 50% 0.5


## Step 7: Score the Model

First the baseline, then accuracy, then the four boxes, then precision and recall.

In [ ]:
accuracy_score(y_test, y_pred) # y-test -> actual answers, y_pred -> predicted answer

0.7950028392958546

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
tn, fp, fn, tp = cm.ravel() # order is fixed

print("Caught (TP):", tp)
print("False alarms (FP):", fp)
print("Missed (FN):", fn)
print("Correct no (TN):", tn)



Caught (TP): 245
False alarms (FP): 139
Missed (FN): 222
Correct no (TN): 1155


In [ ]:
precision_score = tp / (tp+fp)
precision_score

np.float64(0.6380208333333334)

In [ ]:
recall_score = tp / (tp + fn)
recall_score


np.float64(0.5246252676659529)

In [ ]:
# Precision is high -> if your model says churn, high chances of churn
# Recall is low -> Customer actually churning, model can't catch

# Should we increase the threshold of 0.5 or decrease it a bit to improve performance

In [ ]:
scores = model.predict_proba(X_test)[:, 1]
scores

array([0.14978298, 0.05172202, 0.07729034, ..., 0.22594401, 0.30284335,
       0.25648727], shape=(1761,))

In [ ]:
y_pred_03 = (scores >= 0.3).astype(int)

In [ ]:
# The four boxes
tn_03, fp_03, fn_03, tp_03 = confusion_matrix(y_test, y_pred_03).ravel()

# The metrics
accuracy_03 = accuracy_score(y_test, y_pred_03)
precision_03 = tp_03 / (tp_03 + fp_03)
recall_03 = tp_03 / (tp_03 + fn_03)

In [ ]:
accuracy_03

0.7620670073821693

In [ ]:
precision_03

np.float64(0.5355029585798816)

In [ ]:
recall_03

np.float64(0.7751605995717344)

Cost of Mistakes

In [ ]:
# A false alarm: we send an offer to a customer who would have stayed
cost_false_alarm = 500

# A miss: a customer leaves and we did nothing
cost_miss = 6000

# Assumption: an offer to a real leaver keeps them, so it pays for itself.

In [ ]:
bill_05 = fp * cost_false_alarm + fn * cost_miss

print("False alarms:", fp, "  Misses:", fn)
print("Bill at cut-off 0.5: Rs", bill_05)

False alarms: 139   Misses: 222
Bill at cut-off 0.5: Rs 1401500


In [ ]:
bill_03 = fp_03 * cost_false_alarm + fn_03 * cost_miss

print("False alarms:", fp_03, "  Misses:", fn_03)
print("Bill at cut-off 0.5: Rs", bill_03)

False alarms: 314   Misses: 105
Bill at cut-off 0.5: Rs 787000


In [ ]:
# 4 cases
                            Actual Not Churn (-ve)        Actual Churn(+ve)

Predicted
Not Churn(-ve)



Predicted
Churn (+ve)


# Model starts saying everybody will churn
# Will our model catch all the churn customers? -> yes, Recall is high (Out of total actual, how many we caught)
# Will the model be precise? -> Model keeps saying churn, but is it correct very less times > Precision (low)
# Precision -> Out of total that model says are positive, how many were correct
